# Panel de Análisis de Ventas — Mercado Central

Este notebook hace tres cosas, en orden:

1. **Carga** el CSV de detalle de ticket (`ventas_ticket_decimales.csv`).
2. **Calcula** todos los agregados del análisis (productos, cesta de compra, ticket por centro, descuentos, estacionalidad y mix entre centros).
3. **Genera y descarga** el dashboard interactivo en un único archivo HTML, listo para abrir en cualquier navegador.

Ejecuta las celdas en orden, de arriba a abajo.

## 1. Cargar el CSV y hacer una inspección inicial

In [2]:
import pandas as pd
from google.colab import files

# Subir el archivo CSV (se abrirá un selector de archivos)
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Cargar el CSV (separador ';' y decimales con coma, formato europeo)
df = pd.read_csv(filename, sep=';', encoding='utf-8-sig')

# ===== INSPECCIÓN DE LOS DATOS =====
print("=== Dimensiones ===")
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")

print("\n=== Tipos de datos ===")
print(df.dtypes)

print("\n=== Primeras filas ===")
display(df.head())

print("\n=== Valores nulos por columna ===")
print(df.isnull().sum())

print("\n=== Valores únicos en columnas clave ===")
for col in ['nombre_centro', 'nombre_seccion', 'nombre_subseccion', 'tipo_descuento']:
    print(f"{col}: {df[col].nunique()} valores únicos")

ModuleNotFoundError: No module named 'google.colab'

## 2. Calcular todos los agregados del análisis

Reproduce los 6 bloques de análisis que vimos en el dashboard:
productos y secciones, cesta de compra, ticket y horario por centro,
efectividad de descuentos, estacionalidad y mix de producto entre centros.
El resultado se guarda en el diccionario `DATA`.

In [ ]:
import pandas as pd
import numpy as np
import json
from itertools import combinations
from collections import Counter

# ============================================================
# LIMPIEZA Y PREPARACIÓN
# ============================================================
for col in ['precio_unitario', 'precio_final', 'total_linea']:
    df[col] = df[col].astype(str).str.replace(',', '.').astype(float)

df['fecha_dt']    = pd.to_datetime(df['fecha'], format='%d/%m/%Y')
df['fecha_str']   = df['fecha_dt'].dt.strftime('%Y-%m-%d')
df['dia_semana']  = df['fecha_dt'].dt.dayofweek          # 0 = Lunes
df['hora_num']    = df['hora'].str.split(':').str[0].astype(int)
df['tiene_descuento'] = df['descuento_pct'] > 0
df['ahorro']      = (df['precio_unitario'] * df['cantidad']) - df['total_linea']
df['importe_sin_descuento'] = df['precio_unitario'] * df['cantidad']

dias_nombres = ['Lunes','Martes','Miercoles','Jueves','Viernes','Sabado','Domingo']
centros   = sorted(df['nombre_centro'].unique().tolist())
secciones_list = sorted(df['nombre_seccion'].unique().tolist())
fechas    = sorted(df['fecha_str'].unique().tolist())
n_tickets = df['id_ticket'].nunique()

def clean(records, rounds={}):
    """Convierte tipos numpy a tipos nativos de Python y redondea, para que json.dumps no falle."""
    out = []
    for r in records:
        rr = {}
        for k, v in r.items():
            if isinstance(v, (np.integer,)):
                rr[k] = int(v)
            elif isinstance(v, (np.floating,)):
                vv = float(v)
                if abs(vv) < 1e-9:
                    vv = 0.0
                rr[k] = round(vv, rounds.get(k, 2))
            else:
                rr[k] = v
        out.append(rr)
    return out

print(f"Datos preparados: {len(df):,} líneas, {n_tickets:,} tickets, "
      f"{df['nombre_producto'].nunique()} productos, {len(centros)} centros.")

# ============================================================
# KPIs GLOBALES
# ============================================================
ticket_importes = df.groupby('id_ticket')['total_linea'].sum()

kpi_global = {
    "ventas_totales": round(df['total_linea'].sum(), 2),
    "n_tickets": int(n_tickets),
    "n_lineas": int(len(df)),
    "n_productos": int(df['nombre_producto'].nunique()),
    "n_centros": int(df['nombre_centro'].nunique()),
    "n_secciones": int(df['nombre_seccion'].nunique()),
    "ticket_medio": round(ticket_importes.mean(), 2),
    "ticket_mediano": round(ticket_importes.median(), 2),
    "unidades_totales": int(df['cantidad'].sum()),
    "ahorro_total": round(df['ahorro'].sum(), 2),
    "pct_lineas_con_descuento": round(df['tiene_descuento'].mean(), 4),
    "fecha_min": fechas[0],
    "fecha_max": fechas[-1],
}

# ============================================================
# ANÁLISIS 01 — RANKING DE PRODUCTOS, SUBSECCIONES Y SECCIONES
# ============================================================
prod_agg = df.groupby(['nombre_producto','nombre_seccion','nombre_subseccion']).agg(
    unidades=('cantidad','sum'),
    importe=('total_linea','sum'),
    n_lineas=('id_ticket','count'),
    pct_descuento=('tiene_descuento','mean'),
).reset_index()

top_productos_importe    = clean(prod_agg.sort_values('importe',  ascending=False).head(20).to_dict('records'), {'pct_descuento':4})
bottom_productos_importe = clean(prod_agg.sort_values('importe',  ascending=True ).head(20).to_dict('records'), {'pct_descuento':4})
top_productos_unidades   = clean(prod_agg.sort_values('unidades', ascending=False).head(20).to_dict('records'), {'pct_descuento':4})
bottom_productos_unidades= clean(prod_agg.sort_values('unidades', ascending=True ).head(20).to_dict('records'), {'pct_descuento':4})

subsecciones = clean(
    df.groupby(['nombre_subseccion','nombre_seccion']).agg(
        unidades=('cantidad','sum'), importe=('total_linea','sum'),
        n_productos=('nombre_producto','nunique'), n_lineas=('id_ticket','count'),
    ).reset_index().sort_values('importe', ascending=False).to_dict('records')
)

secciones = clean(
    df.groupby('nombre_seccion').agg(
        unidades=('cantidad','sum'), importe=('total_linea','sum'),
        n_productos=('nombre_producto','nunique'), n_subsecciones=('nombre_subseccion','nunique'),
        n_lineas=('id_ticket','count'),
    ).reset_index().sort_values('importe', ascending=False).to_dict('records')
)

# ============================================================
# ANÁLISIS 02 — CESTA DE COMPRA (LIFT)
# ============================================================
ticket_subs = df.groupby('id_ticket')['nombre_subseccion'].apply(lambda x: sorted(set(x)))
sub_freq = Counter()
for subs in ticket_subs:
    sub_freq.update(subs)

pair_counter = Counter()
for subs in ticket_subs:
    if len(subs) > 1:
        for pair in combinations(subs, 2):
            pair_counter[pair] += 1

top_pairs = []
for (a, b), n_ab in pair_counter.most_common(60):
    p_a, p_b, p_ab = sub_freq[a]/n_tickets, sub_freq[b]/n_tickets, n_ab/n_tickets
    lift = p_ab / (p_a * p_b)
    top_pairs.append({"a": a, "b": b, "n_tickets": n_ab, "pct_tickets": round(p_ab*100,2), "lift": round(lift,2)})

top_pairs_by_lift = sorted([p for p in top_pairs if p['n_tickets'] >= 50], key=lambda x: -x['lift'])[:20]
top_pairs_by_freq = sorted(top_pairs, key=lambda x: -x['n_tickets'])[:20]

ticket_secs = df.groupby('id_ticket')['nombre_seccion'].apply(lambda x: sorted(set(x)))
sec_freq = Counter()
for secs in ticket_secs:
    sec_freq.update(secs)
sec_pair_counter = Counter()
for secs in ticket_secs:
    if len(secs) > 1:
        for pair in combinations(secs, 2):
            sec_pair_counter[pair] += 1
sec_pairs = []
for (a, b), n_ab in sec_pair_counter.most_common():
    p_a, p_b, p_ab = sec_freq[a]/n_tickets, sec_freq[b]/n_tickets, n_ab/n_tickets
    lift = p_ab / (p_a * p_b)
    sec_pairs.append({"a": a, "b": b, "n_tickets": n_ab, "pct_tickets": round(p_ab*100,2), "lift": round(lift,2)})
sec_pairs = sorted(sec_pairs, key=lambda x: -x['n_tickets'])

# ============================================================
# ANÁLISIS 03 — TICKET, CESTA Y HORA PICO POR CENTRO
# ============================================================
ticket_info = df.groupby(['id_ticket','nombre_centro']).agg(
    importe=('total_linea','sum'), n_lineas=('nombre_producto','count'),
).reset_index()

por_centro_ticket = clean(
    ticket_info.groupby('nombre_centro').agg(
        n_tickets=('id_ticket','nunique'), ticket_medio=('importe','mean'),
        ticket_mediano=('importe','median'), cesta_media_lineas=('n_lineas','mean'),
    ).reset_index().sort_values('ticket_medio', ascending=False).to_dict('records')
)

hora_centro_df = df.groupby(['nombre_centro','hora_num']).agg(
    importe=('total_linea','sum'), n_tickets=('id_ticket','nunique'),
).reset_index()
hora_centro = clean(hora_centro_df.to_dict('records'))

hora_global = clean(
    df.groupby('hora_num').agg(
        importe=('total_linea','sum'), n_tickets=('id_ticket','nunique'),
    ).reset_index().sort_values('hora_num').to_dict('records')
)

hora_pico_resumen = []
for centro, g in hora_centro_df.groupby('nombre_centro'):
    pico = g.loc[g['importe'].idxmax()]
    hora_pico_resumen.append({"centro": centro, "hora_pico": int(pico['hora_num']), "importe_hora_pico": round(float(pico['importe']),2)})

# ============================================================
# ANÁLISIS 04 — EFECTIVIDAD DE LOS DESCUENTOS
# ============================================================
promo_agg_df = df.groupby('tipo_descuento', dropna=False).agg(
    n_lineas=('id_ticket','count'), unidades=('cantidad','sum'),
    importe_final=('total_linea','sum'), importe_sin_descuento=('importe_sin_descuento','sum'),
    ahorro_total=('ahorro','sum'),
).reset_index()
promo_agg_df['tipo_descuento'] = promo_agg_df['tipo_descuento'].fillna('Sin descuento')
promo_agg_df['pct_ahorro_sobre_pvp'] = promo_agg_df['ahorro_total'] / promo_agg_df['importe_sin_descuento']
promo_agg_df['unidades_por_linea'] = promo_agg_df['unidades'] / promo_agg_df['n_lineas']
promo_analysis = clean(promo_agg_df.sort_values('importe_final', ascending=False).to_dict('records'),
                        {'pct_ahorro_sobre_pvp':4, 'unidades_por_linea':2})

comparativa_descuento = clean(
    df.groupby('tiene_descuento').agg(
        n_lineas=('id_ticket','count'), unidades_media=('cantidad','mean'),
        importe_medio_linea=('total_linea','mean'),
    ).reset_index().to_dict('records'), {'unidades_media':2, 'importe_medio_linea':2}
)

promo_seccion_df = df.groupby('nombre_seccion').agg(
    importe_final=('total_linea','sum'), ahorro_total=('ahorro','sum'),
    pct_lineas_con_descuento=('tiene_descuento','mean'),
).reset_index()
promo_seccion_df['pct_ahorro_sobre_venta'] = promo_seccion_df['ahorro_total'] / promo_seccion_df['importe_final']
promo_seccion = clean(promo_seccion_df.sort_values('ahorro_total', ascending=False).to_dict('records'),
                       {'pct_lineas_con_descuento':4, 'pct_ahorro_sobre_venta':4})

promo_tipo_seccion = clean(
    df[df['tiene_descuento']].groupby(['nombre_seccion','tipo_descuento']).agg(
        n_lineas=('id_ticket','count'), importe=('total_linea','sum'),
    ).reset_index().to_dict('records')
)

# ============================================================
# ANÁLISIS 05 — ESTACIONALIDAD HORARIA Y DIARIA
# ============================================================
serie_diaria_centro = clean(
    df.groupby(['fecha_str','nombre_centro'])['total_linea'].sum().reset_index().to_dict('records')
)
serie_diaria_total = clean(
    df.groupby('fecha_str').agg(importe=('total_linea','sum'), n_tickets=('id_ticket','nunique'))
      .reset_index().to_dict('records')
)

dia_semana_total_df = df.groupby('dia_semana').agg(
    importe=('total_linea','sum'), n_tickets=('id_ticket','nunique'),
).reset_index()
dia_semana_total_df['dia_nombre'] = dia_semana_total_df['dia_semana'].apply(lambda x: dias_nombres[x])
dia_semana_total = clean(dia_semana_total_df.sort_values('dia_semana').to_dict('records'))

dia_semana_centro_df = df.groupby(['nombre_centro','dia_semana']).agg(importe=('total_linea','sum')).reset_index()
dia_semana_centro_df['dia_nombre'] = dia_semana_centro_df['dia_semana'].apply(lambda x: dias_nombres[x])
dia_semana_centro = clean(dia_semana_centro_df.sort_values(['nombre_centro','dia_semana']).to_dict('records'))

hora_dia_semana_df = df.groupby(['dia_semana','hora_num']).agg(importe=('total_linea','sum')).reset_index()
hora_dia_semana_df['dia_nombre'] = hora_dia_semana_df['dia_semana'].apply(lambda x: dias_nombres[x])
hora_dia_semana = clean(hora_dia_semana_df.to_dict('records'))

# ============================================================
# ANÁLISIS 06 — MIX DE PRODUCTO ENTRE CENTROS
# ============================================================
centro_seccion_df = df.groupby(['nombre_centro','nombre_seccion'])['total_linea'].sum().reset_index()
centro_totales = df.groupby('nombre_centro')['total_linea'].sum().to_dict()
centro_seccion_df['pct_del_centro'] = centro_seccion_df.apply(lambda r: r['total_linea']/centro_totales[r['nombre_centro']], axis=1)
centro_seccion = clean(centro_seccion_df.rename(columns={'total_linea':'importe'}).to_dict('records'), {'pct_del_centro':4})

pivot = centro_seccion_df.pivot(index='nombre_seccion', columns='nombre_centro', values='pct_del_centro').fillna(0)
pivot_mix = []
for sec in pivot.index:
    row = {"seccion": sec}
    for centro in pivot.columns:
        row[centro] = round(float(pivot.loc[sec, centro]), 4)
    pivot_mix.append(row)

pivot_rango = pivot.copy()
pivot_rango['rango'] = pivot_rango.max(axis=1) - pivot_rango.min(axis=1)
diffs_sorted = pivot_rango.sort_values('rango', ascending=False)
top_diffs = []
for sec in diffs_sorted.index[:6]:
    row = {"seccion": sec}
    for centro in pivot.columns:
        row[centro] = round(float(diffs_sorted.loc[sec, centro]), 4)
    top_diffs.append(row)

top_prod_centro_df = df.groupby(['nombre_centro','nombre_producto']).agg(
    importe=('total_linea','sum'), unidades=('cantidad','sum')
).reset_index()
top_prod_por_centro = {}
for centro, g in top_prod_centro_df.groupby('nombre_centro'):
    top_prod_por_centro[centro] = clean(g.sort_values('importe', ascending=False).head(5).to_dict('records'))

# ============================================================
# ENSAMBLAJE FINAL: estructura DATA para el dashboard HTML
# ============================================================
DATA = {
    "kpi_global": kpi_global,
    "centros": centros,
    "secciones": secciones,
    "fechas": fechas,
    "top_productos_importe": top_productos_importe,
    "bottom_productos_importe": bottom_productos_importe,
    "top_productos_unidades": top_productos_unidades,
    "bottom_productos_unidades": bottom_productos_unidades,
    "subsecciones": subsecciones,
    "top_pairs_by_freq": top_pairs_by_freq,
    "top_pairs_by_lift": top_pairs_by_lift,
    "sec_pairs": sec_pairs,
    "por_centro_ticket": por_centro_ticket,
    "hora_centro": hora_centro,
    "hora_global": hora_global,
    "hora_pico_resumen": hora_pico_resumen,
    "promo_analysis": promo_analysis,
    "comparativa_descuento": comparativa_descuento,
    "promo_seccion": promo_seccion,
    "promo_tipo_seccion": promo_tipo_seccion,
    "serie_diaria_centro": serie_diaria_centro,
    "serie_diaria_total": serie_diaria_total,
    "dia_semana_total": dia_semana_total,
    "dia_semana_centro": dia_semana_centro,
    "hora_dia_semana": hora_dia_semana,
    "centro_seccion": centro_seccion,
    "pivot_mix": pivot_mix,
    "top_diffs": top_diffs,
    "top_prod_por_centro": top_prod_por_centro,
}

print(f"\nCálculos completados: {len(DATA)} bloques de datos generados.")
print(f"Ventas totales: {kpi_global['ventas_totales']:,.2f} €  |  Ticket medio: {kpi_global['ticket_medio']:.2f} €")


## 3. Generar el dashboard HTML y descargarlo

Esta celda inyecta los datos calculados (`DATA`) en la plantilla del dashboard
y descarga el archivo `.html` final. Es autocontenido: se abre directamente
en cualquier navegador, sin instalar nada.

In [ ]:
import json
from datetime import datetime

# ============================================================
# 1) Plantilla HTML del dashboard (estructura, CSS y placeholders)
# ============================================================
DASHBOARD_HTML_TEMPLATE = '''<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Mercado Central · Panel de Análisis de Ventas — Marzo 2024</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Archivo+Black&family=IBM+Plex+Mono:wght@400;500;600;700&family=Archivo:wght@400;500;600;700;800&display=swap" rel="stylesheet">
<style>
:root{
  --paper:#FAF6EE;
  --paper-dim:#F1EBDB;
  --ink:#1C1B19;
  --ink-soft:#5B5650;
  --green:#2D6A4F;
  --green-soft:#D7E8DD;
  --orange:#E0703A;
  --orange-soft:#FBE3D3;
  --blue:#3D5A80;
  --blue-soft:#DDE6F0;
  --line:#1C1B19;
  --line-soft:rgba(28,27,25,0.14);
  --red:#B3432A;
  --radius:0px;
}
*{box-sizing:border-box;}
html,body{margin:0;padding:0;}
body{
  background:var(--paper);
  color:var(--ink);
  font-family:'Archivo',sans-serif;
  -webkit-font-smoothing:antialiased;
}
.mono{font-family:'IBM Plex Mono',monospace;}

/* ===== Textured backdrop: subtle dot grid like thermal paper ===== */
body::before{
  content:"";
  position:fixed;
  inset:0;
  background-image: radial-gradient(circle, rgba(28,27,25,0.05) 1px, transparent 1px);
  background-size: 14px 14px;
  pointer-events:none;
  z-index:0;
}

.page{
  position:relative;
  z-index:1;
  max-width:1180px;
  margin:0 auto;
  padding:28px 20px 80px;
}

/* ===== HEADER: receipt style ===== */
.receipt-head{
  background:var(--ink);
  color:var(--paper);
  padding:26px 32px 24px;
  position:relative;
  margin-bottom:15px;
}
.receipt-head::after{
  content:"";
  position:absolute;
  left:0; right:0; bottom:-9px;
  height:18px;
  background-image: linear-gradient(135deg, var(--ink) 50%, transparent 50%), linear-gradient(45deg, var(--ink) 50%, transparent 50%);
  background-size: 18px 18px;
  background-position: 0 0;
  background-repeat: repeat-x;
  z-index:2;
}
.head-top{
  display:flex;
  justify-content:space-between;
  align-items:flex-start;
  gap:20px;
  flex-wrap:wrap;
}
.head-brandblock{ display:flex; align-items:center; gap:14px; }
.logo-mark{ flex-shrink:0; }
.eyebrow{
  font-family:'IBM Plex Mono',monospace;
  font-size:11px;
  letter-spacing:.14em;
  text-transform:uppercase;
  color:var(--orange);
  margin:0 0 8px;
}
h1.brand{
  font-family:'Archivo Black',sans-serif;
  font-size:clamp(24px,3.6vw,36px);
  line-height:1.05;
  margin:0 0 4px;
  letter-spacing:-0.01em;
}
.brand-sub{
  font-family:'IBM Plex Mono',monospace;
  font-size:13px;
  color:rgba(250,246,238,0.7);
  margin:0;
  line-height:1.5;
}
.brand-meta{
  margin-top:20px;
  padding-top:16px;
  border-top:1px solid rgba(250,246,238,0.18);
  display:flex;
  flex-wrap:wrap;
  gap:28px;
}
.brand-meta .item{ display:flex; flex-direction:column; gap:3px; }
.brand-meta .item .k{
  font-family:'IBM Plex Mono',monospace;
  font-size:9.5px;
  letter-spacing:.1em;
  text-transform:uppercase;
  color:rgba(250,246,238,0.5);
}
.brand-meta .item .v{
  font-family:'IBM Plex Mono',monospace;
  font-size:13px;
  color:var(--paper);
  font-weight:600;
}

/* ===== BARCODE DIVIDER: separates header from KPI strip ===== */
.barcode-divider{
  background:var(--paper);
  padding:14px 32px 12px;
  margin-top:9px;
  display:flex;
  align-items:center;
  justify-content:space-between;
  gap:18px;
  flex-wrap:wrap;
  border-bottom:1px solid var(--line-soft);
}
.barcode{
  height:30px;
  display:flex;
  align-items:stretch;
  gap:0;
}
.barcode span{ display:block; background:var(--ink); }
.barcode-label{
  font-family:'IBM Plex Mono',monospace;
  font-size:10px;
  letter-spacing:.08em;
  color:var(--ink-soft);
  white-space:nowrap;
}

/* ===== KPI STRIP ===== */
.kpi-strip{
  display:grid;
  grid-template-columns:repeat(6,1fr);
  background:var(--ink);
  border-top:1px dashed rgba(250,246,238,0.25);
}
.kpi{
  padding:16px 14px 18px;
  border-right:1px solid rgba(250,246,238,0.16);
  color:var(--paper);
}
.kpi:last-child{border-right:none;}
.kpi .label{
  font-family:'IBM Plex Mono',monospace;
  font-size:10px;
  letter-spacing:.1em;
  text-transform:uppercase;
  color:rgba(250,246,238,0.55);
  margin:0 0 6px;
}
.kpi .value{
  font-family:'IBM Plex Mono',monospace;
  font-weight:700;
  font-size:clamp(15px,2vw,22px);
  color:var(--paper);
  white-space:nowrap;
}
.kpi .value.green{color:#8FD9B6;}
.kpi .value.orange{color:#F4A876;}

/* ===== NAV TABS ===== */
.tabs{
  display:flex;
  flex-wrap:wrap;
  gap:0;
  margin:28px 0 0;
  border-bottom:2px solid var(--ink);
}
.tab-btn{
  font-family:'IBM Plex Mono',monospace;
  font-size:12.5px;
  letter-spacing:.03em;
  background:transparent;
  border:none;
  border-top:2px solid transparent;
  border-left:1px solid var(--line-soft);
  padding:13px 16px 11px;
  cursor:pointer;
  color:var(--ink-soft);
  display:flex;
  align-items:center;
  gap:8px;
  transition:background .15s ease, color .15s ease;
}
.tab-btn:first-child{border-left:none;}
.tab-btn .num{
  font-weight:700;
  color:var(--orange);
}
.tab-btn:hover{ background:var(--paper-dim); color:var(--ink); }
.tab-btn.active{
  background:var(--ink);
  color:var(--paper);
  border-top-color:var(--orange);
}
.tab-btn.active .num{ color:var(--orange); }

/* ===== PANELS ===== */
.panel{ display:none; padding-top:26px; animation:fadein .25s ease; }
.panel.active{ display:block; }
@keyframes fadein{ from{opacity:0; transform:translateY(4px);} to{opacity:1; transform:translateY(0);} }

.panel-head{ margin-bottom:18px; }
.panel-head .kicker{
  font-family:'IBM Plex Mono',monospace;
  font-size:11px;
  letter-spacing:.14em;
  text-transform:uppercase;
  color:var(--green);
  margin:0 0 6px;
}
.panel-head h2{
  font-family:'Archivo',sans-serif;
  font-weight:800;
  font-size:clamp(20px,2.6vw,28px);
  margin:0 0 6px;
  letter-spacing:-0.01em;
}
.panel-head p.desc{
  font-size:14px;
  color:var(--ink-soft);
  max-width:720px;
  margin:0;
  line-height:1.5;
}

.grid{
  display:grid;
  gap:18px;
}
.grid-2{ grid-template-columns:1.3fr 1fr; }
.grid-3{ grid-template-columns:repeat(3,1fr); }
.grid-2-eq{ grid-template-columns:1fr 1fr; }

.card{
  background:#fff;
  border:1px solid var(--line-soft);
  padding:18px 18px 14px;
  position:relative;
}
.card.tinted{ background:var(--paper-dim); }
.card-title{
  font-family:'IBM Plex Mono',monospace;
  font-size:11.5px;
  letter-spacing:.06em;
  text-transform:uppercase;
  color:var(--ink-soft);
  margin:0 0 12px;
  display:flex;
  justify-content:space-between;
  align-items:baseline;
}
.card-title .tag{
  font-size:10px;
  background:var(--green-soft);
  color:var(--green);
  padding:2px 7px;
  letter-spacing:0;
  text-transform:none;
}
.chart-wrap{ position:relative; width:100%; }
.chart-wrap.h220{ height:220px; }
.chart-wrap.h260{ height:260px; }
.chart-wrap.h300{ height:300px; }
.chart-wrap.h340{ height:340px; }

/* ===== FILTER BAR ===== */
.filterbar{
  display:flex;
  flex-wrap:wrap;
  gap:10px;
  align-items:center;
  margin-bottom:18px;
  padding:12px 14px;
  background:var(--blue-soft);
  border:1px solid rgba(61,90,128,0.25);
}
.filterbar .flabel{
  font-family:'IBM Plex Mono',monospace;
  font-size:10.5px;
  letter-spacing:.08em;
  text-transform:uppercase;
  color:var(--blue);
  margin-right:2px;
}
select.fselect{
  font-family:'IBM Plex Mono',monospace;
  font-size:12.5px;
  padding:6px 10px;
  border:1px solid var(--blue);
  background:#fff;
  color:var(--ink);
  cursor:pointer;
}
select.fselect:focus{ outline:2px solid var(--orange); outline-offset:1px; }

/* ===== TABLE ===== */
table.dtable{
  width:100%;
  border-collapse:collapse;
  font-size:13px;
}
table.dtable th{
  font-family:'IBM Plex Mono',monospace;
  font-size:10.5px;
  letter-spacing:.05em;
  text-transform:uppercase;
  text-align:left;
  color:var(--ink-soft);
  border-bottom:2px solid var(--ink);
  padding:7px 8px;
}
table.dtable td{
  padding:7px 8px;
  border-bottom:1px solid var(--line-soft);
  font-family:'IBM Plex Mono',monospace;
  font-size:12.5px;
}
table.dtable tr:hover td{ background:var(--paper-dim); }
table.dtable td.num{ text-align:right; }
.bar-cell{ display:flex; align-items:center; gap:8px; }
.bar-track{ flex:1; height:7px; background:var(--paper-dim); position:relative; min-width:40px;}
.bar-fill{ position:absolute; left:0; top:0; height:100%; background:var(--green); }
.bar-fill.orange{ background:var(--orange); }

/* ===== INSIGHT BOX ===== */
.insight{
  border-left:4px solid var(--orange);
  background:var(--orange-soft);
  padding:12px 16px;
  font-size:13.5px;
  line-height:1.55;
  margin-bottom:18px;
}
.insight b{ color:var(--red); }
.insight.green{ border-left-color:var(--green); background:var(--green-soft); }
.insight.green b{ color:var(--green); }
.insight .ico{
  font-family:'IBM Plex Mono',monospace;
  font-size:10px;
  letter-spacing:.1em;
  text-transform:uppercase;
  display:block;
  margin-bottom:4px;
  color:var(--ink-soft);
}

.legend-row{ display:flex; flex-wrap:wrap; gap:14px; margin-top:10px; font-size:11px; font-family:'IBM Plex Mono',monospace; color:var(--ink-soft); }
.legend-row .dot{ display:inline-block; width:9px; height:9px; margin-right:5px; vertical-align:middle; }

footer.foot{
  margin-top:50px;
  padding-top:18px;
  border-top:1px dashed var(--line-soft);
  font-family:'IBM Plex Mono',monospace;
  font-size:11px;
  color:var(--ink-soft);
  display:flex;
  justify-content:space-between;
  flex-wrap:wrap;
  gap:6px;
}

@media (max-width: 880px){
  .grid-2, .grid-2-eq, .grid-3{ grid-template-columns:1fr; }
  .kpi-strip{ grid-template-columns:repeat(2,1fr); }
  .kpi{ border-bottom:1px solid rgba(250,246,238,0.16); }
}
</style>
</head>
<body>
<div class="page">

  <div class="receipt-head">
    <div class="head-top">
      <div class="head-brandblock">
        <svg class="logo-mark" width="52" height="52" viewBox="0 0 56 56" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Logotipo Mercado Central">
          <circle cx="28" cy="28" r="27" fill="#FAF6EE"/>
          <circle cx="28" cy="28" r="27" fill="none" stroke="#1C1B19" stroke-width="1.4"/>
          <circle cx="28" cy="28" r="22" fill="#1C1B19"/>
          <path d="M15.5 21 L15.5 17 Q18.375 14 21.25 17 Q24.125 14 27 17 Q29.875 14 32.75 17 Q35.625 14 38.5 17 L38.5 21 Z" fill="#E0703A"/>
          <line x1="15.5" y1="21" x2="38.5" y2="21" stroke="#FAF6EE" stroke-width="0.8"/>
          <text x="28" y="40" font-family="Archivo Black, sans-serif" font-size="15" font-weight="900" fill="#FAF6EE" text-anchor="middle" letter-spacing="0.5">MC</text>
        </svg>
        <div>
          <p class="eyebrow">Mercado Central · Dirección Comercial</p>
          <h1 class="brand">Panel de Análisis de Ventas</h1>
        </div>
      </div>
    </div>
    <p class="brand-sub">Detalle transaccional a nivel de ticket y línea de producto · Informe interno</p>
    <div class="brand-meta">
      <div class="item"><span class="k">Periodo analizado</span><span class="v">01/03/2024 – 01/04/2024</span></div>
      <div class="item"><span class="k">Centros</span><span class="v">4</span></div>
      <div class="item"><span class="k">Referencias</span><span class="v">282 productos</span></div>
      <div class="item"><span class="k">Fuente</span><span class="v">Detalle de ticket (POS)</span></div>
    </div>
  </div>

  <div class="barcode-divider">
    <div class="barcode" id="barcode"></div>
    <span class="barcode-label">REF. INFORME · MC-2024-03 · GENERADO A PARTIR DE 153.008 LÍNEAS DE TICKET</span>
  </div>

  <div class="kpi-strip" id="kpiStrip"></div>

  <nav class="tabs" id="tabs"></nav>

  <main id="panels"></main>

  <footer class="foot">
    <span>Mercado Central · Panel de Análisis de Ventas · Marzo 2024</span>
    <span>Fuente: detalle de ticket (153.008 líneas) · Datos de carácter ilustrativo</span>
  </footer>

</div>

<script id="data-json" type="application/json">
__DATA_JSON__
</script>
<script>
__APP_JS__
</script>
</body>
</html>
'''

# ============================================================
# 2) JavaScript de la aplicación (lógica de los 6 paneles)
# ============================================================
DASHBOARD_APP_JS = '''// ============================================================
// CONFIG & HELPERS
// ============================================================
const DATA = JSON.parse(document.getElementById('data-json').textContent);

const COLORS = {
  green:'#2D6A4F', greenSoft:'#D7E8DD',
  orange:'#E0703A', orangeSoft:'#FBE3D3',
  blue:'#3D5A80', blueSoft:'#DDE6F0',
  red:'#B3432A', ink:'#1C1B19', inkSoft:'#5B5650', paper:'#FAF6EE', paperDim:'#F1EBDB'
};
const CENTRO_COLORS = {};
(DATA.centros||[]).forEach((c,i)=>{
  CENTRO_COLORS[c] = [COLORS.green, COLORS.orange, COLORS.blue, COLORS.red][i % 4];
});

function fmtEUR(v, dec=0){
  return new Intl.NumberFormat('es-ES',{minimumFractionDigits:dec,maximumFractionDigits:dec}).format(v) + ' €';
}
function fmtNum(v, dec=0){
  return new Intl.NumberFormat('es-ES',{minimumFractionDigits:dec,maximumFractionDigits:dec}).format(v);
}
function fmtPct(v, dec=1){
  return new Intl.NumberFormat('es-ES',{minimumFractionDigits:dec,maximumFractionDigits:dec}).format(v*100) + '%';
}
function shortCentro(c){ return c.replace('Centro ',''); }
function cap(s){ return s.charAt(0).toUpperCase()+s.slice(1); }
function safeChart(canvasId, config){
  if(typeof Chart === 'undefined') return null;
  const el = document.getElementById(canvasId);
  if(!el) return null;
  try { return new Chart(el, config); }
  catch(e){ console.error('Chart error on', canvasId, e); return null; }
}

if(typeof Chart !== 'undefined'){
  Chart.defaults.font.family = "'IBM Plex Mono', monospace";
  Chart.defaults.font.size = 11;
  Chart.defaults.color = COLORS.inkSoft;
  Chart.defaults.borderColor = 'rgba(28,27,25,0.08)';
  Chart.defaults.plugins.legend.labels.boxWidth = 12;
  Chart.defaults.plugins.legend.labels.usePointStyle = false;
}

// ============================================================
// BARCODE DIVIDER
// ============================================================
(function renderBarcode(){
  const el = document.getElementById('barcode');
  if(!el) return;
  // Patrón determinista (no decorativo-aleatorio puro) basado en el total de líneas del dataset,
  // para que el ancho de barras varíe de forma reproducible.
  const seed = DATA.kpi_global.n_lineas || 153008;
  let html = '';
  const nBars = 46;
  for(let i=0;i<nBars;i++){
    const w = 1 + ((seed >> (i % 12)) + i * 7) % 4; // 1..4 px
    const gapBig = (i % 6 === 0);
    html += `<span style="width:${w}px;margin-right:${gapBig?3:1}px;"></span>`;
  }
  el.innerHTML = html;
})();

// ============================================================
// KPI STRIP
// ============================================================
(function renderKPIs(){
  const k = DATA.kpi_global;
  const items = [
    {label:'Ventas totales', value: fmtEUR(k.ventas_totales), cls:''},
    {label:'Tickets', value: fmtNum(k.n_tickets), cls:''},
    {label:'Ticket medio', value: fmtEUR(k.ticket_medio,2), cls:'green'},
    {label:'Unidades vendidas', value: fmtNum(k.unidades_totales), cls:''},
    {label:'% líneas con descuento', value: fmtPct(k.pct_lineas_con_descuento), cls:'orange'},
    {label:'Ahorro concedido', value: fmtEUR(k.ahorro_total), cls:'orange'},
  ];
  document.getElementById('kpiStrip').innerHTML = items.map(it=>`
    <div class="kpi">
      <p class="label">${it.label}</p>
      <p class="value ${it.cls}">${it.value}</p>
    </div>`).join('');
})();

// ============================================================
// TAB DEFINITIONS
// ============================================================
const TABS = [
  {id:'productos', num:'01', label:'Productos'},
  {id:'cesta', num:'02', label:'Cesta de compra'},
  {id:'centros', num:'03', label:'Ticket &amp; horario'},
  {id:'promos', num:'04', label:'Descuentos'},
  {id:'tiempo', num:'05', label:'Estacionalidad'},
  {id:'mix', num:'06', label:'Mix entre centros'},
];

document.getElementById('tabs').innerHTML = TABS.map((t,i)=>
  `<button class="tab-btn ${i===0?'active':''}" data-tab="${t.id}"><span class="num">${t.num}</span> ${t.label}</button>`
).join('');

document.querySelectorAll('.tab-btn').forEach(btn=>{
  btn.addEventListener('click', ()=>{
    document.querySelectorAll('.tab-btn').forEach(b=>b.classList.remove('active'));
    document.querySelectorAll('.panel').forEach(p=>p.classList.remove('active'));
    btn.classList.add('active');
    document.getElementById('panel-'+btn.dataset.tab).classList.add('active');
  });
});

const panelsRoot = document.getElementById('panels');
TABS.forEach((t,i)=>{
  const div = document.createElement('div');
  div.className = 'panel' + (i===0?' active':'');
  div.id = 'panel-'+t.id;
  panelsRoot.appendChild(div);
});

// ============================================================
// PANEL 01 — PRODUCTOS (ranking unidades / importe)
// ============================================================
function renderProductos(){
  const root = document.getElementById('panel-productos');
  root.innerHTML = `
    <div class="panel-head">
      <p class="kicker">Análisis 01</p>
      <h2>¿Qué productos y subsecciones más venden — y cuáles apenas se mueven?</h2>
      <p class="desc">Ranking por importe y por unidades. Las referencias de mayor peso orientan las decisiones de espacio en tienda y nivel de stock; la cola de baja rotación señala candidatas a revisión o retirada del surtido.</p>
    </div>

    <div class="filterbar">
      <span class="flabel">Ordenar por</span>
      <select class="fselect" id="prodSortBy">
        <option value="importe">Importe (€)</option>
        <option value="unidades">Unidades</option>
      </select>
      <span class="flabel">Mostrar</span>
      <select class="fselect" id="prodDir">
        <option value="top">Top 12</option>
        <option value="bottom">Cola 12 (menos venden)</option>
      </select>
    </div>

    <div class="grid grid-2">
      <div class="card">
        <p class="card-title">Ranking de productos <span class="tag" id="prodTag">Top por importe</span></p>
        <div class="chart-wrap h340"><canvas id="chartProductos"></canvas></div>
      </div>
      <div class="card">
        <p class="card-title">Importe por subsección <span class="tag">Top 12</span></p>
        <div class="chart-wrap h340"><canvas id="chartSubsecciones"></canvas></div>
      </div>
    </div>

    <div class="insight">
      <span class="ico">Lectura</span>
      El top de importe lo dominan productos de <b>carne (Ternera)</b>, <b>bebidas (Alcohol)</b> y <b>charcutería (Jamón)</b> — productos de precio unitario alto, no necesariamente los más comprados en unidades. La subsección <b>Ternera</b> es la que más factura con solo 6 referencias.
    </div>

    <div class="card">
      <p class="card-title">Detalle por sección · unidades, importe y nº de referencias</p>
      <div id="tablaSecciones"></div>
    </div>
  `;

  let chartProd, chartSub;

  function draw(){
    const sortBy = document.getElementById('prodSortBy').value;
    const dir = document.getElementById('prodDir').value;
    const key = sortBy === 'importe' ? (dir==='top'?'top_productos_importe':'bottom_productos_importe')
                                      : (dir==='top'?'top_productos_unidades':'bottom_productos_unidades');
    const rows = DATA[key].slice(0,12);
    document.getElementById('prodTag').textContent = (dir==='top'?'Top 12 · ':'Cola 12 · ') + (sortBy==='importe'?'por importe':'por unidades');

    const labels = rows.map(r=>r.nombre_producto);
    const vals = rows.map(r=> sortBy==='importe' ? r.importe : r.unidades);

    if(chartProd) chartProd.destroy();
    chartProd = safeChart('chartProductos', {
      type:'bar',
      data:{ labels, datasets:[{
        data: vals,
        backgroundColor: dir==='top'? COLORS.green : COLORS.orange,
        borderRadius:0,
      }]},
      options:{
        indexAxis:'y',
        plugins:{ legend:{display:false}, tooltip:{ callbacks:{ label: c=> sortBy==='importe' ? fmtEUR(c.parsed.x,2) : fmtNum(c.parsed.x)+' ud' } } },
        scales:{ x:{ grid:{color:'rgba(28,27,25,0.08)'}, ticks:{ callback:v=> sortBy==='importe'? (v/1000)+'k€' : v } }, y:{ ticks:{font:{size:10}} } }
      }
    });
  }

  document.getElementById('prodSortBy').addEventListener('change', draw);
  document.getElementById('prodDir').addEventListener('change', draw);
  draw();

  const subRows = DATA.subsecciones.slice(0,12);
  chartSub = safeChart('chartSubsecciones', {
    type:'bar',
    data:{ labels: subRows.map(r=>r.nombre_subseccion+' ('+r.nombre_seccion+')'), datasets:[{
      data: subRows.map(r=>r.importe), backgroundColor: COLORS.blue, borderRadius:0
    }]},
    options:{
      indexAxis:'y',
      plugins:{ legend:{display:false}, tooltip:{ callbacks:{ label:c=>fmtEUR(c.parsed.x,2) } } },
      scales:{ x:{ ticks:{ callback:v=>(v/1000)+'k€' } }, y:{ ticks:{font:{size:10}} } }
    }
  });

  // Tabla secciones
  const maxImporte = Math.max(...DATA.secciones.map(s=>s.importe));
  const tbl = `<table class="dtable">
    <thead><tr><th>Sección</th><th class="num">Importe</th><th>% del total</th><th class="num">Unidades</th><th class="num">Refs.</th></tr></thead>
    <tbody>
      ${DATA.secciones.map(s=>`
        <tr>
          <td>${cap(s.nombre_seccion)}</td>
          <td class="num">${fmtEUR(s.importe)}</td>
          <td><div class="bar-cell"><div class="bar-track"><div class="bar-fill" style="width:${(s.importe/maxImporte*100).toFixed(1)}%"></div></div><span style="font-family:'IBM Plex Mono',monospace;font-size:11px;min-width:30px;">${(s.importe/DATA.kpi_global.ventas_totales*100).toFixed(1)}%</span></div></td>
          <td class="num">${fmtNum(s.unidades)}</td>
          <td class="num">${s.n_productos}</td>
        </tr>`).join('')}
    </tbody>
  </table>`;
  document.getElementById('tablaSecciones').innerHTML = tbl;
}

// ============================================================
// PANEL 02 — CESTA DE COMPRA
// ============================================================
function renderCesta(){
  const root = document.getElementById('panel-cesta');
  root.innerHTML = `
    <div class="panel-head">
      <p class="kicker">Análisis 02</p>
      <h2>¿Qué productos se compran juntos?</h2>
      <p class="desc">Medido con <b>lift</b>: si lift &gt; 1, esas dos categorías aparecen juntas en un ticket más a menudo de lo que pasaría por puro azar. Lift = 1 significa independencia total (se compran juntas justo lo esperado, ni más ni menos).</p>
    </div>

    <div class="insight">
      <span class="ico">Hallazgo</span>
      Los valores de <b>lift se mantienen todos muy cerca de 1</b> (entre 1.0 y 1.16). Esto indica que, en el periodo analizado, <b>no existen afinidades de cesta relevantes</b>: comprar vino, por ejemplo, no aumenta de forma apreciable la probabilidad de comprar aperitivos. Las combinaciones más frecuentes reflejan simplemente categorías de compra habitual (fruta, lácteos, bebidas) presentes en la mayoría de los tickets, no una asociación real entre ellas.
    </div>

    <div class="grid grid-2">
      <div class="card">
        <p class="card-title">Pares de subsecciones más frecuentes <span class="tag">% de tickets que llevan ambas</span></p>
        <div class="chart-wrap h340"><canvas id="chartCestaFreq"></canvas></div>
      </div>
      <div class="card">
        <p class="card-title">Pares con mayor asociación (lift) <span class="tag">soporte mín. 50 tickets</span></p>
        <div class="chart-wrap h340"><canvas id="chartCestaLift"></canvas></div>
      </div>
    </div>

    <div class="card" style="margin-top:18px;">
      <p class="card-title">Matriz de co-ocurrencia entre secciones <span class="tag">% de tickets con ambas secciones</span></p>
      <div id="heatmapSecciones" class="chart-wrap" style="height:420px; overflow-x:auto;"></div>
    </div>
  `;

  const freqRows = DATA.top_pairs_by_freq.slice(0,10);
  safeChart('chartCestaFreq', {
    type:'bar',
    data:{ labels: freqRows.map(r=>r.a+' + '+r.b), datasets:[{ data: freqRows.map(r=>r.pct_tickets), backgroundColor: COLORS.green, borderRadius:0 }]},
    options:{ indexAxis:'y', plugins:{legend:{display:false}, tooltip:{callbacks:{label:c=>c.parsed.x.toFixed(1)+'% de tickets'}}}, scales:{ x:{ ticks:{callback:v=>v+'%'} }, y:{ticks:{font:{size:9.5}}} } }
  });

  const liftRows = DATA.top_pairs_by_lift.slice(0,10);
  safeChart('chartCestaLift', {
    type:'bar',
    data:{ labels: liftRows.map(r=>r.a+' + '+r.b), datasets:[{ data: liftRows.map(r=>r.lift), backgroundColor: COLORS.orange, borderRadius:0 }]},
    options:{ indexAxis:'y', plugins:{legend:{display:false}, tooltip:{callbacks:{label:c=>'Lift: '+c.parsed.x.toFixed(2)}}}, scales:{ x:{ min:0.9, title:{display:true,text:'Lift (1.0 = independencia)'} }, y:{ticks:{font:{size:9.5}}} } }
  });

  // Heatmap secciones x secciones via canvas grid using HTML table (más legible que canvas)
  const secciones = DATA.secciones.map(s=>s.nombre_seccion);
  const pairMap = {};
  DATA.sec_pairs.forEach(p=>{ pairMap[p.a+'|'+p.b]=p.pct_tickets; pairMap[p.b+'|'+p.a]=p.pct_tickets; });
  let maxV = Math.max(...DATA.sec_pairs.map(p=>p.pct_tickets));

  function colorFor(v){
    if(v===undefined) return '#fff';
    const t = v / maxV;
    // interpolate paperDim -> green
    const r1=241,g1=235,b1=219, r2=45,g2=106,b2=79;
    const r = Math.round(r1+(r2-r1)*t), g = Math.round(g1+(g2-g1)*t), b = Math.round(b1+(b2-b1)*t);
    return `rgb(${r},${g},${b})`;
  }
  let html = '<table class="dtable" style="font-size:10px;"><thead><tr><th></th>' +
    secciones.map(s=>`<th style="writing-mode:vertical-rl;text-orientation:mixed;height:90px;">${s}</th>`).join('') + '</tr></thead><tbody>';
  secciones.forEach(rowSec=>{
    html += `<tr><th style="text-align:right;">${rowSec}</th>`;
    secciones.forEach(colSec=>{
      if(rowSec===colSec){ html += `<td style="background:${COLORS.ink};color:${COLORS.paper};text-align:center;">—</td>`; }
      else {
        const v = pairMap[rowSec+'|'+colSec];
        html += `<td style="background:${colorFor(v)};text-align:center;color:${v && v/maxV>0.6?'#fff':COLORS.ink}">${v?v.toFixed(0):'0'}</td>`;
      }
    });
    html += '</tr>';
  });
  html += '</tbody></table>';
  document.getElementById('heatmapSecciones').innerHTML = html;
}

// ============================================================
// PANEL 03 — TICKET MEDIO / CESTA / HORA PICO POR CENTRO
// ============================================================
function renderCentros(){
  const root = document.getElementById('panel-centros');
  root.innerHTML = `
    <div class="panel-head">
      <p class="kicker">Análisis 03</p>
      <h2>Ticket medio, tamaño de cesta y hora pico por centro</h2>
      <p class="desc">Comparativa operativa de los 4 centros: cuánto gasta cada cliente, cuántas líneas compra de media, y a qué hora se concentran las ventas — clave para dimensionar cajas y personal.</p>
    </div>

    <div class="grid grid-3">
      ${DATA.por_centro_ticket.map(c=>`
        <div class="card tinted">
          <p class="card-title">${shortCentro(c.nombre_centro)}</p>
          <p style="font-family:'IBM Plex Mono',monospace;font-size:26px;font-weight:700;margin:0 0 2px;color:${CENTRO_COLORS[c.nombre_centro]};">${fmtEUR(c.ticket_medio,2)}</p>
          <p style="font-size:11px;color:var(--ink-soft);margin:0 0 10px;">ticket medio (mediana ${fmtEUR(c.ticket_mediano,2)})</p>
          <p style="font-size:12px;margin:2px 0;"><b>${c.n_tickets.toLocaleString('es-ES')}</b> tickets</p>
          <p style="font-size:12px;margin:2px 0;"><b>${c.cesta_media_lineas.toFixed(1)}</b> líneas/ticket de media</p>
        </div>
      `).join('')}
    </div>

    <div class="grid grid-2" style="margin-top:18px;">
      <div class="card">
        <p class="card-title">Ventas por hora del día <span class="tag" id="horaTag">Todos los centros</span></p>
        <div class="filterbar" style="margin-bottom:12px;">
          <span class="flabel">Centro</span>
          <select class="fselect" id="centroHoraFilter">
            <option value="ALL">Todos (agregado)</option>
            ${DATA.centros.map(c=>`<option value="${c}">${shortCentro(c)}</option>`).join('')}
          </select>
        </div>
        <div class="chart-wrap h300"><canvas id="chartHoraCentro"></canvas></div>
      </div>
      <div class="card">
        <p class="card-title">Hora pico de ventas por centro</p>
        <div class="chart-wrap h300"><canvas id="chartHoraPico"></canvas></div>
      </div>
    </div>

    <div class="insight green" style="margin-top:18px;">
      <span class="ico">Lectura</span>
      El ticket medio es muy similar entre centros (66–68 €) y la cesta también (~11,5 líneas). La diferencia operativa relevante está en <b>cuándo</b> se concentra la afluencia: Valencia-Este registra su pico por la tarde (19h), mientras el resto concentra la demanda al mediodía.
    </div>
  `;

  let chartHC;
  function drawHoraCentro(){
    const sel = document.getElementById('centroHoraFilter').value;
    let labels, data, color;
    if(sel==='ALL'){
      labels = DATA.hora_global.map(h=>h.hora_num+'h');
      data = DATA.hora_global.map(h=>h.importe);
      color = COLORS.ink;
    } else {
      const rows = DATA.hora_centro.filter(h=>h.nombre_centro===sel).sort((a,b)=>a.hora_num-b.hora_num);
      labels = rows.map(h=>h.hora_num+'h');
      data = rows.map(h=>h.importe);
      color = CENTRO_COLORS[sel];
    }
    if(chartHC) chartHC.destroy();
    chartHC = safeChart('chartHoraCentro', {
      type:'line',
      data:{ labels, datasets:[{ data, borderColor:color, backgroundColor:color+'33', fill:true, tension:.35, pointRadius:3 }]},
      options:{ plugins:{legend:{display:false}, tooltip:{callbacks:{label:c=>fmtEUR(c.parsed.y,0)}}}, scales:{ y:{ ticks:{callback:v=>(v/1000)+'k€'} } } }
    });
  }
  document.getElementById('centroHoraFilter').addEventListener('change', drawHoraCentro);
  drawHoraCentro();

  safeChart('chartHoraPico', {
    type:'bar',
    data:{ labels: DATA.hora_pico_resumen.map(h=>shortCentro(h.centro)), datasets:[{
      data: DATA.hora_pico_resumen.map(h=>h.hora_pico),
      backgroundColor: DATA.hora_pico_resumen.map(h=>CENTRO_COLORS[h.centro]),
      borderRadius:0,
    }]},
    options:{ plugins:{legend:{display:false}, tooltip:{callbacks:{label:c=>'Hora pico: '+c.parsed.y+':00h'}}}, scales:{ y:{ title:{display:true,text:'Hora del día'}, min:0, max:23 } } }
  });
}

// ============================================================
// PANEL 04 — DESCUENTOS
// ============================================================
function renderPromos(){
  const root = document.getElementById('panel-promos');
  root.innerHTML = `
    <div class="panel-head">
      <p class="kicker">Análisis 04</p>
      <h2>¿Qué promoción mueve más volumen — y cuánto cuesta en descuento?</h2>
      <p class="desc">El detalle de ticket no incluye el coste de producto, por lo que no es posible calcular el margen real de cada promoción. El indicador disponible con precisión es el <b>ahorro concedido al cliente</b> por tipo de promoción, y si ese ahorro se traduce en un mayor volumen de unidades por línea.</p>
    </div>

    <div class="grid grid-2">
      <div class="card">
        <p class="card-title">Importe vendido por tipo de descuento</p>
        <div class="chart-wrap h300"><canvas id="chartPromoImporte"></canvas></div>
      </div>
      <div class="card">
        <p class="card-title">Ahorro total concedido por tipo</p>
        <div class="chart-wrap h300"><canvas id="chartPromoAhorro"></canvas></div>
      </div>
    </div>

    <div class="insight">
      <span class="ico">Hallazgo</span>
      Las unidades medias por línea son <b>prácticamente idénticas</b> con descuento (1,82) y sin descuento (1,82). En el periodo analizado, <b>el descuento no incentiva un mayor volumen de compra</b>: únicamente reduce el precio pagado por la misma cantidad. Si el objetivo comercial de las promociones es elevar el volumen, este indicador sugiere que no se está logrando.
    </div>

    <div class="grid grid-2">
      <div class="card">
        <p class="card-title">% de ahorro sobre venta, por sección</p>
        <div class="chart-wrap h300"><canvas id="chartPromoSeccion"></canvas></div>
      </div>
      <div class="card">
        <p class="card-title">Comparativa: líneas con vs. sin descuento</p>
        <div id="tablaComparativa"></div>
      </div>
    </div>
  `;

  const promo = DATA.promo_analysis;
  safeChart('chartPromoImporte', {
    type:'bar',
    data:{ labels: promo.map(p=>p.tipo_descuento), datasets:[{ data: promo.map(p=>p.importe_final), backgroundColor: COLORS.green, borderRadius:0 }]},
    options:{ plugins:{legend:{display:false}, tooltip:{callbacks:{label:c=>fmtEUR(c.parsed.y,0)}}}, scales:{ y:{ ticks:{callback:v=>(v/1000)+'k€'} }, x:{ticks:{font:{size:9.5}}} } }
  });

  safeChart('chartPromoAhorro', {
    type:'bar',
    data:{ labels: promo.filter(p=>p.tipo_descuento!=='Sin descuento').map(p=>p.tipo_descuento),
      datasets:[{ data: promo.filter(p=>p.tipo_descuento!=='Sin descuento').map(p=>p.ahorro_total), backgroundColor: COLORS.orange, borderRadius:0 }]},
    options:{ plugins:{legend:{display:false}, tooltip:{callbacks:{label:c=>fmtEUR(c.parsed.y,0)+' de ahorro'}}}, scales:{ y:{ ticks:{callback:v=>(v/1000)+'k€'} } } }
  });

  const ps = DATA.promo_seccion;
  safeChart('chartPromoSeccion', {
    type:'bar',
    data:{ labels: ps.map(p=>cap(p.nombre_seccion)), datasets:[{ data: ps.map(p=>p.pct_ahorro_sobre_venta*100), backgroundColor: COLORS.blue, borderRadius:0 }]},
    options:{ indexAxis:'y', plugins:{legend:{display:false}, tooltip:{callbacks:{label:c=>c.parsed.x.toFixed(1)+'%'}}}, scales:{ x:{ ticks:{callback:v=>v+'%'} }, y:{ticks:{font:{size:10}}} } }
  });

  const comp = DATA.comparativa_descuento;
  const sinD = comp.find(c=>c.tiene_descuento===false);
  const conD = comp.find(c=>c.tiene_descuento===true);
  document.getElementById('tablaComparativa').innerHTML = `
    <table class="dtable">
      <thead><tr><th></th><th class="num">Sin descuento</th><th class="num">Con descuento</th></tr></thead>
      <tbody>
        <tr><td>Nº de líneas</td><td class="num">${fmtNum(sinD.n_lineas)}</td><td class="num">${fmtNum(conD.n_lineas)}</td></tr>
        <tr><td>Unidades medias / línea</td><td class="num">${sinD.unidades_media.toFixed(2)}</td><td class="num">${conD.unidades_media.toFixed(2)}</td></tr>
        <tr><td>Importe medio / línea</td><td class="num">${fmtEUR(sinD.importe_medio_linea,2)}</td><td class="num">${fmtEUR(conD.importe_medio_linea,2)}</td></tr>
      </tbody>
    </table>
    <p style="font-size:12px;color:var(--ink-soft);margin-top:10px;line-height:1.5;">El <b>${fmtPct(DATA.kpi_global.pct_lineas_con_descuento)}</b> de las líneas de ticket llevan algún tipo de promoción aplicada.</p>
  `;
}

// ============================================================
// PANEL 05 — ESTACIONALIDAD
// ============================================================
function renderTiempo(){
  const root = document.getElementById('panel-tiempo');
  root.innerHTML = `
    <div class="panel-head">
      <p class="kicker">Análisis 05</p>
      <h2>Estacionalidad horaria y diaria dentro del mes</h2>
      <p class="desc">¿Qué días y horas vende más cada centro? Útil para turnos de personal, reposición de stock y planificación de promociones puntuales.</p>
    </div>

    <div class="card">
      <p class="card-title">Ventas diarias · marzo 2024 <span class="tag">por centro</span></p>
      <div class="chart-wrap h300"><canvas id="chartSerieDiaria"></canvas></div>
      <div class="legend-row">
        ${DATA.centros.map(c=>`<span><span class="dot" style="background:${CENTRO_COLORS[c]}"></span>${shortCentro(c)}</span>`).join('')}
      </div>
    </div>

    <div class="grid grid-2" style="margin-top:18px;">
      <div class="card">
        <p class="card-title">Ventas por día de la semana</p>
        <div class="chart-wrap h280"><canvas id="chartDiaSemana"></canvas></div>
      </div>
      <div class="card">
        <p class="card-title">Mapa de calor: hora × día de semana</p>
        <div id="heatmapHoraDia" style="overflow-x:auto;"></div>
      </div>
    </div>

    <div class="insight green" style="margin-top:18px;">
      <span class="ico">Lectura</span>
      <b>Viernes, sábado y domingo</b> concentran la mayor facturación de la semana (más de 135.000 € cada uno), frente a martes-miércoles-jueves (106.000–109.000 €). El <b>lunes también repunta</b> con fuerza, un patrón compatible con compra de reposición tras el fin de semana, aunque el dato no permite confirmar la causa.
    </div>
  `;

  // serie diaria por centro
  const fechas = [...new Set(DATA.serie_diaria_centro.map(r=>r.fecha_str))].sort();
  const datasets = DATA.centros.map(c=>{
    const map = {};
    DATA.serie_diaria_centro.filter(r=>r.nombre_centro===c).forEach(r=>map[r.fecha_str]=r.total_linea);
    return {
      label: shortCentro(c),
      data: fechas.map(f=>map[f]||0),
      borderColor: CENTRO_COLORS[c],
      backgroundColor: CENTRO_COLORS[c]+'22',
      tension:.3, pointRadius:1.5, borderWidth:2,
    };
  });
  safeChart('chartSerieDiaria', {
    type:'line',
    data:{ labels: fechas.map(f=>f.slice(8,10)+'/'+f.slice(5,7)), datasets },
    options:{ plugins:{legend:{display:false}, tooltip:{callbacks:{label:c=>c.dataset.label+': '+fmtEUR(c.parsed.y,0)}}}, scales:{ y:{ticks:{callback:v=>(v/1000)+'k€'}}, x:{ticks:{maxTicksLimit:16, font:{size:9}}} } }
  });

  const dw = DATA.dia_semana_total;
  safeChart('chartDiaSemana', {
    type:'bar',
    data:{ labels: dw.map(d=>d.dia_nombre), datasets:[{ data: dw.map(d=>d.importe),
      backgroundColor: dw.map(d=> ['Viernes','Sabado','Domingo'].includes(d.dia_nombre)? COLORS.green : COLORS.blue), borderRadius:0 }]},
    options:{ plugins:{legend:{display:false}, tooltip:{callbacks:{label:c=>fmtEUR(c.parsed.y,0)}}}, scales:{ y:{ticks:{callback:v=>(v/1000)+'k€'}} } }
  });

  // heatmap hora x dia
  const dias = ['Lunes','Martes','Miercoles','Jueves','Viernes','Sabado','Domingo'];
  const horas = [...new Set(DATA.hora_dia_semana.map(h=>h.hora_num))].sort((a,b)=>a-b);
  const hdMap = {};
  let maxHD = 0;
  DATA.hora_dia_semana.forEach(h=>{ hdMap[h.dia_nombre+'|'+h.hora_num]=h.importe; if(h.importe>maxHD) maxHD=h.importe; });
  function colorForHD(v){
    const t = (v||0)/maxHD;
    const r1=241,g1=235,b1=219, r2=224,g2=112,b2=58;
    const r = Math.round(r1+(r2-r1)*t), g = Math.round(g1+(g2-g1)*t), b = Math.round(b1+(b2-b1)*t);
    return `rgb(${r},${g},${b})`;
  }
  let html = '<table class="dtable" style="font-size:10px;"><thead><tr><th>Día</th>' + horas.map(h=>`<th>${h}h</th>`).join('') + '</tr></thead><tbody>';
  dias.forEach(d=>{
    html += `<tr><th style="text-align:left;">${d.slice(0,3)}</th>`;
    horas.forEach(h=>{
      const v = hdMap[d+'|'+h];
      html += `<td style="background:${colorForHD(v)};text-align:center;color:${v/maxHD>0.6?'#fff':COLORS.ink}">${v?Math.round(v/100):''}</td>`;
    });
    html += '</tr>';
  });
  html += '</tbody></table><p style="font-size:10.5px;color:var(--ink-soft);margin-top:8px;">Valores en cientos de €</p>';
  document.getElementById('heatmapHoraDia').innerHTML = html;
}

// ============================================================
// PANEL 06 — MIX DE PRODUCTO ENTRE CENTROS
// ============================================================
function renderMix(){
  const root = document.getElementById('panel-mix');
  root.innerHTML = `
    <div class="panel-head">
      <p class="kicker">Análisis 06</p>
      <h2>Comparativa del mix de producto entre centros</h2>
      <p class="desc">Comparativa del peso (%) de cada sección dentro de las ventas totales de cada centro. Si el mix es muy parecido entre centros, la política comercial puede ser uniforme; si difiere mucho, conviene surtido diferenciado.</p>
    </div>

    <div class="card">
      <p class="card-title">Peso de cada sección dentro de cada centro <span class="tag">% sobre ventas del centro</span></p>
      <div class="chart-wrap h340"><canvas id="chartMixRadar"></canvas></div>
    </div>

    <div class="insight green" style="margin-top:18px;">
      <span class="ico">Lectura</span>
      El mix de producto es <b>muy homogéneo</b> entre los 4 centros: ninguna sección varía más de 1,5 puntos porcentuales de peso entre el centro que más y el que menos la vende. No se observa evidencia de especialización geográfica relevante en el periodo analizado.
    </div>

    <div class="grid grid-2" style="margin-top:18px;">
      <div class="card">
        <p class="card-title">Secciones con mayor diferencia de peso entre centros</p>
        <div class="chart-wrap h300"><canvas id="chartDiffs"></canvas></div>
      </div>
      <div class="card">
        <p class="card-title">Top producto por centro <span class="tag" id="mixTag">Centro Madrid-Norte</span></p>
        <div class="filterbar" style="margin-bottom:12px;">
          <span class="flabel">Centro</span>
          <select class="fselect" id="mixCentroFilter">
            ${DATA.centros.map(c=>`<option value="${c}">${shortCentro(c)}</option>`).join('')}
          </select>
        </div>
        <div id="tablaTopCentro"></div>
      </div>
    </div>
  `;

  // Radar: secciones (categorias) x centro (datasets) - usar % del mix
  const seccionesNames = DATA.pivot_mix.map(p=>cap(p.seccion));
  const radarDatasets = DATA.centros.map(c=>({
    label: shortCentro(c),
    data: DATA.pivot_mix.map(p=>(p[c]||0)*100),
    borderColor: CENTRO_COLORS[c],
    backgroundColor: CENTRO_COLORS[c]+'18',
    pointRadius:2, borderWidth:2,
  }));
  safeChart('chartMixRadar', {
    type:'radar',
    data:{ labels: seccionesNames, datasets: radarDatasets },
    options:{
      plugins:{ legend:{ position:'bottom', labels:{font:{size:10}} }, tooltip:{callbacks:{label:c=>c.dataset.label+': '+c.parsed.r.toFixed(1)+'%'}} },
      scales:{ r:{ ticks:{ callback:v=>v+'%', font:{size:9} }, pointLabels:{font:{size:9.5}} } }
    }
  });

  safeChart('chartDiffs', {
    type:'bar',
    data:{ labels: DATA.top_diffs.map(d=>cap(d.seccion)), datasets: DATA.centros.map(c=>({
      label: shortCentro(c), data: DATA.top_diffs.map(d=>(d[c]||0)*100), backgroundColor: CENTRO_COLORS[c]
    })) },
    options:{ plugins:{ legend:{ position:'bottom', labels:{font:{size:9.5}} } }, scales:{ x:{ ticks:{font:{size:9}} }, y:{ ticks:{callback:v=>v+'%'} } } }
  });

  function drawTopCentro(){
    const centro = document.getElementById('mixCentroFilter').value;
    document.getElementById('mixTag').textContent = shortCentro(centro);
    const rows = DATA.top_prod_por_centro[centro] || [];
    document.getElementById('tablaTopCentro').innerHTML = `
      <table class="dtable">
        <thead><tr><th>Producto</th><th class="num">Importe</th><th class="num">Unidades</th></tr></thead>
        <tbody>
          ${rows.map(r=>`<tr><td>${r.nombre_producto}</td><td class="num">${fmtEUR(r.importe,2)}</td><td class="num">${fmtNum(r.unidades)}</td></tr>`).join('')}
        </tbody>
      </table>
    `;
  }
  document.getElementById('mixCentroFilter').addEventListener('change', drawTopCentro);
  drawTopCentro();
}

// ============================================================
// INIT — cada panel se aísla: un fallo en uno no debe tumbar el resto
// ============================================================
if(typeof Chart === 'undefined'){
  const warn = document.createElement('div');
  warn.style.cssText = 'background:#FBE3D3;border-left:4px solid #E0703A;padding:10px 16px;margin:0 0 18px;font-family:IBM Plex Mono,monospace;font-size:12.5px;';
  warn.textContent = 'No se ha podido cargar la librería de gráficos (Chart.js) desde el CDN. Comprueba la conexión a internet y vuelve a abrir el archivo; el resto del contenido (KPIs, tablas y rankings) sigue disponible.';
  document.getElementById('panels').before(warn);
}
[renderProductos, renderCesta, renderCentros, renderPromos, renderTiempo, renderMix].forEach(fn=>{
  try { fn(); } catch(e){ console.error('Error renderizando panel:', fn.name, e); }
});
'''

# ============================================================
# 3) Ensamblaje: inyectar los datos calculados (DATA) y el JS
# ============================================================
data_json_str = json.dumps(DATA, ensure_ascii=False)

html_final = DASHBOARD_HTML_TEMPLATE.replace('__DATA_JSON__', data_json_str)
html_final = html_final.replace('__APP_JS__', DASHBOARD_APP_JS)

assert '__DATA_JSON__' not in html_final, "Quedó un placeholder de datos sin sustituir"
assert '__APP_JS__' not in html_final, "Quedó un placeholder de JS sin sustituir"

# ============================================================
# 4) Guardar y descargar
# ============================================================
nombre_archivo = "dashboard_ventas_marzo2024.html"

with open(nombre_archivo, 'w', encoding='utf-8') as f:
    f.write(html_final)

print(f"Dashboard generado: {nombre_archivo}")
print(f"Tamaño del archivo: {len(html_final)/1024:.1f} KB")
print(f"Generado el: {datetime.now().strftime('%d/%m/%Y %H:%M')}")

try:
    from google.colab import files
    files.download(nombre_archivo)
    print("\nDescarga iniciada. Si el navegador la bloquea, revisa el icono de descargas bloqueadas en la barra de direcciones.")
except ImportError:
    print("\nNo se detectó el entorno de Colab; el archivo ha quedado guardado en el directorio actual.")
